# Õppetund 01 - Sissejuhatus tehisintellekti agentidesse

Tere tulemast esimesse õppetundi kursusel **AI Agents for Beginners**!

**AI agent** on programm, mis kasutab oma mõtlemismootorina suurt keelemudelit (LLM) ja suudab võtta *tegevusi* reaalses maailmas — kutsudes API-sid, pärides andmebaase või käivitades koodi — eesmärgiga täita kasutaja eest seatud ülesanne.

Selles märkmes ehitad oma esimese agendi: **Reisiagendi**, mis soovitab puhkuse sihtkohti. Samal ajal õpid, kuidas:

1. Ühenduda Microsoft Foundry Agent Service'iga kasutades **Microsoft Agent Frameworki**.
2. Anda agendile **tööriist** — lihtne Pythoni funktsioon, mida ta saab kutsuda.
3. Käivitada agent ja uurida tema vastust.
4. Jooksutada agenti ja kuvada vastus token-tokeni kaupa.


## Seadistamine

Enne selle märkmiku käivitamist veendu, et sul on:

1. **Microsoft Foundry projekt**, millel on kasutusele võetud vestlusmudel (nt `gpt-5-mini`).
2. **Sisselogimine Azure CLI-ga** — käivita terminalis käsk `az login`.
3. **Määra vajalikud keskkonnamuutujad:**
   - `AZURE_AI_PROJECT_ENDPOINT` — sinu Microsoft Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — kasutusele võetud mudeli nimi.

Allolev lahter installib vajalikud Python paketid.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Oma esimese agendi loomine

Agendil on vaja kahte asja:

- **Juhised**, mis ütlevad talle *kes ta on* ja *kuidas käituda* (süsteemi prompt).
- **Vahendid** — Python-funktsioonid, mis on kaunistatud `@tool` ja mida agent saab kutsuda info toomiseks või toimingute tegemiseks.

Allpool määratleme lihtsa tööriista, mis tagastab nimekirja populaarsetest puhkuse sihtkohtadest. Agent kasutab seda tööriista, kui kasutaja palub reisisoovitusi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Voogedastuse vastused

Interaktiivsema kogemuse jaoks saate agendi vastuse **voogedastada**. Täieliku vastuse ootamise asemel annab agent tekstitükke, kui need tekivad. See on eriti kasulik vestlusliidestes, kus soovite väljundit reaalajas kuvada.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

- **Luua pakkuja**, mis ühendub Microsoft Foundry Agent Service'iga `FoundryChatClient`i kaudu.
- **Määratleda tööriist** kasutades `@tool` dekoratiivi, et agent saaks kutsuda teie Python'i funktsioone.
- **Käivitada agent** kasutaja sõnumiga ja printida selle vastust.
- **Voo vastuseid** reaalajas väljundi jaoks.

Järgmises õppetükis uurime agentide raamistikke põhjalikumalt ning õpime, kuidas anda agentidele võimsamaid tööriistu ja mitmeastmelise mõtlemise võimeid.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
